In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from matplotlib import pyplot as plt

In [ ]:
np.random.seed(42)

## Preprocessing for Logistic Regression

Preprocessing steps for the logistic regression model includes:

- Transformation: Since `title` and `content` are both text-based data, they must be transformed into vectors using TF-IDF. 
    - Normally, I would also be encoding `type` using One Hot Encoding which would ascribe each type category a number from 0 to 11. However, sklearn handles target feature transformation under the hood.



# Handling Missing Titles
The dataset has 35 rows with a missing title (roughly 0.12% of the total dataset). I considered using a `SimpleImputer` to handle this which would replace the NaN values with empty strings, but this introduced issues when used alongside `ColumnTransformer` due to `SimpleImputer`'s 2-D input requirement. 

Instead of dropping these 35 rows from the dataset I decided to manually fill them with empty Strings, as can be seen below.

In [ ]:
news = pd.read_csv("../datasets/audited_data.csv")
news["title"] = news["title"].fillna("")

In [ ]:
X = news[["title", "content"]]
y = news["type"]

y.value_counts()

# Preprocessing Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

print(X_train.shape)
print(y_train.shape)

In [ ]:
preprocessor = ColumnTransformer([
    ("title", TfidfVectorizer(), "title"),
    ("content", TfidfVectorizer(), "content")
])

# Hyperparameter Tuning (Logistic Regression)

In [ ]:
metrics = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

In [ ]:
lr_tuning = LogisticRegression(
    max_iter = 1000,
    C = 1.5
)

In [ ]:
lr_tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lr_tuning),
])


In [ ]:
stratified_kfold_cv = StratifiedKFold(
    n_splits = 3,
    shuffle = True,
    random_state = 42
)

In [ ]:
results = cross_validate(
    lr_tuning_pipeline,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics,
    return_train_score = True
    )

In [ ]:
for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")

In [ ]:
lr = LogisticRegression(
    max_iter = 1000
)

In [ ]:
stratified_kfold_cv = StratifiedKFold(
    n_splits = 10,
    shuffle = True,
    random_state = 42
)

In [ ]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lr)
])

In [ ]:
results = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics,
    return_train_score = True
    )

In [ ]:
for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")

In [ ]:
lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average = 'macro', zero_division = 0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average = 'macro', zero_division = 0):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred, average = 'macro', zero_division = 0):.4f}")

In [ ]:
labels = lr.classes_

cm = confusion_matrix(y_test, y_pred)

display = ConfusionMatrixDisplay(
    confusion_matrix = cm,
    display_labels = labels
    )

display.plot()

plt.xticks(rotation = 45)
plt.title('Logistic Regression (C = 1.0)')
plt.show()


## Decision Tree 

In [ ]:
dt = DecisionTreeClassifier()

In [ ]:
pipeline_dt = Pipeline ([
    ("preprocessor", preprocessor),
    ("classifier", dt)
])

In [ ]:
pipeline_dt.fit(X_train, y_train)
y_pred = pipeline_dt.predict(X_test)
print(f"held-out f1_macro: {f1_score(
    y_test,
    y_pred,
    average = "macro"
):.4f}")

In [ ]:
results = cross_validate(
    pipeline_dt,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics,
    return_train_score = True
)

In [ ]:
for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")

In [ ]:
print(dt.get_depth())

In [ ]:
lr_tuningC = LogisticRegression(
    max_iter = 1000,
    C = 1.5,
    random_state = 42
)

pipeline_lr_tuning = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lr_tuningC),
])

results = cross_validate(
    pipeline_lr_tuning,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics,
    return_train_score = True
    )

for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")

In [ ]:
dt_tuning_md = DecisionTreeClassifier(
    max_depth = 50
)

pipeline_dt_tuning = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", dt_tuning_md),
])

results = cross_validate(
    pipeline_dt_tuning,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics,
    return_train_score = True
    )

for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators = 10
)

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf)
])

In [ ]:
rf_cross_validator = StratifiedKFold(
    n_splits = 3,
    shuffle = True,
    random_state = 42
)

In [ ]:
rf_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv = rf_cross_validator,
    scoring = metrics,
    return_train_score = True
)

In [ ]:
for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")